# 第7章：打造聊天應用程式
## OpenAI API 快速入門

本筆記本改編自包含可存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務的[Azure OpenAI 範例資源庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)。

Python OpenAI API 也能與 Azure OpenAI 模型搭配使用，只需做些微調整。詳細了解差異請參考：[如何使用 Python 在 OpenAI 與 Azure OpenAI 端點間切換](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# 概覽  
「大型語言模型是將文本映射到文本的函數。給定一個輸入的文本字串，大型語言模型嘗試預測接下來會出現的文本」(1)。這個「快速入門」筆記本將介紹使用者高階的 LLM 概念、開始使用 AML 所需的核心套件需求、對提示設計的簡單介紹，以及幾個不同使用案例的簡短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立您的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 憑證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文字](#1.-summarize-text)  
[2. 分類文字](#2.-classify-text)  
[3. 產生新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示  
這個簡短的練習將提供如何向 OpenAI 模型提交提示以執行簡單任務「摘要」的基本介紹。


<strong>步驟</strong>：  
1. 在你的 Python 環境中安裝 OpenAI 函式庫  
2. 載入標準輔助函式庫，並設定你為所建立的 OpenAI 服務所使用的典型安全認證  
3. 為你的任務選擇模型  
4. 為模型建立一個簡單的提示  
5. 向模型 API 提交你的請求！


### 1. 安裝 OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. 匯入輔助函式庫並實例化憑證


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. 找到合適的模型  
像 GPT-4o 和 GPT-4o mini 這樣的模型能理解和生成自然語言，並且在 OpenAI 平台上提供，具有不同的運算能力和速度，適合不同的任務。 


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. 提示設計  

「大型語言模型的神奇之處在於，透過在大量文本上訓練以最小化此預測誤差，模型最終學會了對這些預測有用的概念。例如，它們學會了像」（1）：

* 如何拼寫
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編碼
* 等等。

#### 如何控制大型語言模型  
「在所有輸入到大型語言模型的內容中，最具影響力的莫過於文本提示（prompt）」（1）。

大型語言模型可以通過幾種方式被提示產生輸出：

指令：告訴模型你想要什麼
補全：誘導模型完成你想要的開頭部分
示範：向模型展示你想要什麼，透過：
提示中的幾個範例
訓練數據集中數百或數千個範例進行微調」



#### 建立提示的三大基本指南：

<strong>展示並說明</strong>。透過指令、範例或兩者結合清楚表達你想要的。如果你想讓模型將一個項目清單按字母順序排列或對段落進行情感分類，就要讓它知道這就是你的需求。

<strong>提供高品質資料</strong>。如果你想建立分類器或讓模型遵循某種模式，確保有足夠的範例。務必校對你的範例——模型通常足夠聰明能挑出基本的拼寫錯誤並給出回應，但它也可能假設這是故意的，進而影響回應。

**檢查你的設置。** 溫度（temperature）和 top_p 設定控制模型生成回應的確定性。如果你要求的回應只有一個正確答案，則應將這些設定調低；如果你想要更具多樣性的回應，則可調高。人們對這些設定最常犯的錯誤是誤以為它們是「機智」或「創造力」的控制。


來源：https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重複相同的呼叫，結果會有什麼不同？ 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 摘要文本  
#### 挑戰  
透過在文本段落末尾添加“tl;dr:”來總結文本。請注意模型如何理解在無需額外指示下執行多項任務。您可以嘗試使用比 tl;dr 更具描述性的提示，以調整模型的行為並自訂所獲得的摘要(3)。  

近期研究顯示，先以大型語料庫進行預訓練，接著在特定任務上進行微調，在多項自然語言處理任務和基準測試中帶來明顯提升。雖然此方法在架構上通常與任務無關，但仍需數千至數萬個特定任務的微調數據集。相較之下，人類通常只需少數範例或簡單指令便能執行新的語言任務——這是當前自然語言處理系統仍然難以達到的目標。本文展示，擴大語言模型規模可大幅提升與任務無關的少量範例學習表現，有時甚至能達到與先前最先進微調方法相當的競爭力。 



摘要  


# 幾種使用案例練習  
1. 文本摘要  
2. 文本分類  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 分類文字  
#### 挑戰  
將項目分類到推論時提供的類別。在以下範例中，我們在提示中同時提供類別和要分類的文字(*playground_reference)。 

客戶詢問：您好，我筆記型電腦鍵盤上的一個鍵最近壞了，我需要更換：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 產生新產品名稱
#### 挑戰
從示例詞中創建產品名稱。我們在提示中包含了關於我們要為其生成名稱的產品的資訊。我們還提供了一個相似的示例來展示我們希望收到的模式。我們也將溫度值設置得較高，以增加隨機性和更多創新的回應。

產品描述：家庭用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙適合任何腳尺寸的鞋子。
種子詞：適應性、合腳、全方位合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文字的最佳實務](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 更多幫助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
